# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys, hashlib, tempfile, subprocess, json
import pandas as pd
import numpy as np
from IPython.display import display

CURRENT_DIR = Path.cwd()
roots = [CURRENT_DIR, *CURRENT_DIR.parents]
STUDY_ROOT = next((p for p in roots if (p / 'chapter05/chapter05.ipynb').is_file()), None)
if STUDY_ROOT is None:
    raise FileNotFoundError('chapter05/chapter05.ipynb가 포함된 과제 저장소에서 실행하세요')
course_candidates = [STUDY_ROOT.parent / 'llm-data-analysis-course']
PROJECT_ROOT = next((p for p in course_candidates if (p / 'data/processed').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('인접 수업 폴더의 Chapter 05 processed 데이터가 필요합니다')
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
CHAPTER_DIR = STUDY_ROOT / 'chapter06'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'chapter06'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('입력: 인접 수업 폴더 data/processed, Chapter 05 처리 결과')
print('출력: 수업 작업 폴더 reports/chapter06 (로컬 분석 산출물)')
print('Python:', sys.version.split()[0], 'pandas:', pd.__version__)


입력: 인접 수업 폴더 data/processed, Chapter 05 처리 결과
출력: 수업 작업 폴더 reports/chapter06 (로컬 분석 산출물)
Python: 3.12.14 pandas: 3.0.5


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


### 이 제출본의 실행 기준

- 과제 저장소의 chapter06 폴더 또는 저장소 루트에서 실행하며 인접 llm-data-analysis-course의 data/processed 네 파일을 읽음
- 입력은 Chapter 05 전용 practice/chapter05/data/raw로 처리한 결과이고, 필요하면 먼저 같은 과제 저장소의 Chapter 05 Notebook을 실행함
- 아래 출처 검증은 Chapter 05 Notebook의 함수 정의와 전용 Raw도 사용함, 일반 공통 Raw를 처리한 파일과 섞지 않음
- 함수·스크립트 실행은 이 Notebook에 포함한 정의로 수행하고 생성 결과는 수업 폴더 reports/chapter06에 저장함


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')

customers = customers.drop(columns=['name'], errors='ignore')


전처리 데이터 불러오기 완료


In [3]:
# Chapter 05 Notebook 함수로 임시 생성하여 현재 processed 파일의 출처 확인
raw_dir = PROJECT_ROOT / 'practice/chapter05/data/raw'
with tempfile.TemporaryDirectory(prefix='ch06-lineage-') as td:
    tmp = Path(td)
    ch05 = json.loads((STUDY_ROOT / 'chapter05/chapter05.ipynb').read_text(encoding='utf-8'))
    function_source = next(''.join(c['source']) for c in ch05['cells'] if c['cell_type'] == 'code' and ''.join(c['source']).startswith('# Chapter 05 전처리 함수:')).split('module_raw_data =', 1)[0]
    script_path = tmp / 'preprocess.py'
    script_path.write_text(function_source + '\nimport sys\nsave_processed_data(preprocess_sales_data(load_sales_data(Path(sys.argv[1]))), Path(sys.argv[2]))\n', encoding='utf-8')
    run = subprocess.run([sys.executable, str(script_path), str(raw_dir), str(tmp / 'processed')], capture_output=True, text=True)
    if run.returncode: raise RuntimeError('Chapter 05 재현 실패')
    checks = []
    for path in required_files:
        current_hash = hashlib.sha256(path.read_bytes()).hexdigest()
        regenerated_hash = hashlib.sha256((tmp / 'processed' / path.name).read_bytes()).hexdigest()
        checks.append({'file': path.name, 'sha256': current_hash, 'Chapter05 일치': current_hash == regenerated_hash})
    display(pd.DataFrame(checks))
    assert all(row['Chapter05 일치'] for row in checks)


,file,sha256,Chapter05 일치
0,customers_clean.csv,f14a59af4a8a900c0b9f72c7fab3367df2ba1e1419f24f...,True
1,products_clean.csv,b6a1f8263f815c9990f01049f87c17f9d45ed6b45d4583...,True
2,orders_clean.csv,b7264a74270919ae0b9c1d7bc999f7c0f472d97dfdf445...,True
3,order_items_clean.csv,82389ef808e4f9db56ad76e52fe511b0face3f016cb817...,True


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [4]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 5)
columns: ['customer_id', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [5]:
date_checks = []
for name, df, col in [('orders', orders, 'order_date'), ('customers', customers, 'signup_date')]:
    before = df[col].notna()
    converted = pd.to_datetime(df[col], errors='coerce')
    date_checks.append({'dataset': name, 'original_missing': int((~before).sum()),
                        'new_conversion_failures': int((before & converted.isna()).sum()),
                        'total_missing': int(converted.isna().sum()),
                        'min': converted.min(), 'max': converted.max()})
    df[col] = converted
orders['order_month'] = orders['order_date'].dt.to_period('M').astype('string')
display(pd.DataFrame(date_checks))


,dataset,original_missing,new_conversion_failures,total_missing,min,max
0,orders,3,0,3,2025-07-01,2028-12-31
1,customers,2,0,2,2023-07-21,2026-08-24


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [6]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'orders, order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","orders, order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [7]:
customers['age'].describe()


count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

### 고객 나이 해석

- 평균 나이는 약 43.91세이나 최댓값 150세와 Chapter 05 중앙값 대체가 포함되어 있음
- 연령대별 구매 차이를 해석하기 전 이상값과 대체 기준을 확인할 필요가 있음


In [8]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [9]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [10]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,뷰티,17,17.71
1,패션,16,16.67
2,식품,16,16.67
3,전자기기,14,14.58
4,도서,13,13.54
5,스포츠,10,10.42
6,생활용품,10,10.42


In [11]:
products['price'].describe()


count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [12]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
1,뷰티,17,672294.0,18000.0,9999999.0
4,식품,16,120812.0,11000.0,195000.0
6,패션,16,115125.0,28000.0,197000.0
5,전자기기,14,113143.0,21000.0,178000.0
0,도서,13,103692.0,34000.0,182000.0
3,스포츠,10,78400.0,5000.0,183000.0
2,생활용품,10,75000.0,29000.0,147000.0


### 상품 가격 해석

- 뷰티 평균 표시가격 약 672,294원은 최고 표시가격 9,999,999원의 영향을 받음
- 표시가격 평균을 실제 판매 단가로 사용하지 않고 주문 상세 금액/수량과 구분함


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [13]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


### 주문 상태 해석

- 전체 300건 중 completed 180건·cancelled 63건·refunded 57건임
- 금액 분석에서는 completed 상세만 사용하며 완료 주문 중 상세가 없는 주문은 별도로 확인함


In [14]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [15]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [16]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


### 금액 분석 범위와 연결 기준

- 주문 상태 분포는 전체 주문으로 확인하고 아래 금액 집계는 completed 주문 상세로 제한함
- 상품·날짜·고객 연결 실패는 미상 그룹으로 남겨 금액 누락을 확인함


In [17]:
# 금액 분석은 completed 주문의 상세만 사용함. 주문 상태 분포는 전체 주문 기준 유지
for name, df, key in [('customers', customers, 'customer_id'), ('products', products, 'product_id'), ('orders', orders, 'order_id')]:
    assert df[key].notna().all() and not df[key].duplicated().any(), f'{name} 키 오류'
assert np.allclose(order_items['line_total'], order_items['quantity'] * order_items['unit_price'])
completed_orders = orders.loc[orders['order_status'].eq('completed')].copy()
completed_items = order_items.merge(completed_orders[['order_id']], on='order_id', how='inner', validate='many_to_one')
completed_total = completed_items['line_total'].sum()
print('전체 상세의 미연결 주문:', int((~order_items['order_id'].isin(orders['order_id'])).sum()))
print('전체 주문의 미연결 고객:', int((~orders['customer_id'].isin(customers['customer_id'])).sum()))
print('전체 상세의 미연결 상품:', int((~order_items['product_id'].isin(products['product_id'])).sum()))
print('completed 주문:', len(completed_orders), '상세가 있는 고유 주문:', completed_items['order_id'].nunique())
print('completed 상세:', len(completed_items), '원본 금액:', f'{completed_total:,.0f}')
print('completed 기간:', completed_orders['order_date'].min(), '~', completed_orders['order_date'].max())


전체 상세의 미연결 주문: 2
전체 주문의 미연결 고객: 2
전체 상세의 미연결 상품: 26
completed 주문: 180 상세가 있는 고유 주문: 179
completed 상세: 446 원본 금액: 141,340,000
completed 기간: 2025-07-02 00:00:00 ~ 2027-01-15 00:00:00


In [18]:
sales_items = completed_items.merge(products, on='product_id', how='left', validate='many_to_one', indicator=True)
print('병합 전/후:', len(completed_items), len(sales_items))
print('미연결 상품 상세:', int(sales_items['_merge'].eq('left_only').sum()))
print('미연결 상품 금액:', sales_items.loc[sales_items['_merge'].eq('left_only'), 'line_total'].sum())
sales_items = sales_items.drop(columns='_merge')
sales_items['category'] = sales_items['category'].fillna('미연결 상품')
sales_items.head()


병합 전/후: 446 446
미연결 상품 상세: 16
미연결 상품 금액: 4015000.0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [19]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
7,패션,290.0,32940000.0,23.31
5,식품,226.0,25035000.0,17.71
6,전자기기,181.0,21590000.0,15.28
2,뷰티,180.0,17751000.0,12.56
0,도서,174.0,17663000.0,12.50
4,스포츠,165.0,13237000.0,9.37
3,생활용품,125.0,9109000.0,6.44
1,미연결 상품,34.0,4015000.0,2.84


In [20]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,290.0,32940000.0,23.31,115125.0
1,식품,226.0,25035000.0,17.71,120812.0
2,전자기기,181.0,21590000.0,15.28,113143.0
3,뷰티,180.0,17751000.0,12.56,672294.0
4,도서,174.0,17663000.0,12.50,103692.0
5,스포츠,165.0,13237000.0,9.37,78400.0
6,생활용품,125.0,9109000.0,6.44,75000.0
7,미연결 상품,34.0,4015000.0,2.84,NaN


### 카테고리 금액 해석

- 패션 금액 32,940,000원은 전체 completed 금액의 23.31%로 가장 큼
- 미연결 상품 4,015,000원도 분모에 포함했으며, 상품 수·실제 판매 단가 비교는 아래 과제 2에서 확인함


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [21]:
order_sales = completed_items.merge(completed_orders, on='order_id', how='left', validate='many_to_one')
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype('string').fillna('날짜 미상')
print('병합 전/후:', len(completed_items), len(order_sales))
print('날짜 미상 상세:', int(order_sales['order_date'].isna().sum()))
print('날짜 미상 금액:', order_sales.loc[order_sales['order_date'].isna(), 'line_total'].sum())
order_sales.head()


병합 전/후: 446 446
날짜 미상 상세: 6
날짜 미상 금액: 1903000.0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [22]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,9561000.0,11,869182.0
1,2025-08,9362000.0,17,550706.0
2,2025-09,5299000.0,4,1324750.0
3,2025-10,12629000.0,17,742882.0
4,2025-11,9583000.0,16,598938.0
5,2025-12,8581000.0,12,715083.0
6,2026-01,8135000.0,9,903889.0
7,2026-02,7689000.0,12,640750.0
8,2026-03,7247000.0,13,557462.0
9,2026-04,11459000.0,14,818500.0


### 월별 금액 해석

- 2026-08은 16,980,000원·15건이며 평균 주문 금액은 1,132,000원임
- 날짜 미상은 별도 그룹으로 보존하고, 2027년 주문일과 월별 관측 기간을 확인하기 전 장기 추세나 원인을 단정하지 않음


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [23]:
customer_sales_base = order_sales.merge(customers, on='customer_id', how='left', validate='many_to_one', indicator=True)
print('미연결 고객 상세:', int(customer_sales_base['_merge'].eq('left_only').sum()))
customer_sales_base['city'] = customer_sales_base['city'].fillna('미연결 고객')
customer_sales = (customer_sales_base.groupby(['customer_id', 'city'], as_index=False, dropna=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False))
customer_sales['avg_order_value'] = (customer_sales['total_sales'] / customer_sales['order_count']).round(0)
customer_sales.head(10)


미연결 고객 상세: 1


,customer_id,city,order_count,total_sales,avg_order_value
1,3,서울,3,6702000.0,2234000.0
7,14,대구,4,5204000.0,1301000.0
57,90,수원,3,4311000.0,1437000.0
59,93,울산,4,4001000.0,1000250.0
47,65,광주,6,3977000.0,662833.0
66,104,서울,2,3966000.0,1983000.0
11,20,성남,4,3808000.0,952000.0
27,38,성남,4,3656000.0,914000.0
19,28,인천,4,3201000.0,800250.0
13,22,부산,3,3109000.0,1036333.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [24]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [25]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

[path.name for path in REPORT_DIR.glob('ch06_*.csv')]


['ch06_monthly_sales.csv',
 'ch06_product_category.csv',
 'ch06_eda_questions.csv',
 'ch06_category_sales.csv',
 'ch06_customer_city.csv',
 'ch06_customer_sales.csv']

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [26]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전체 고객·상품·주문의 기본 분포와 completed 주문 상세의 금액을 탐색했습니다. 상품·고객 미연결 및 날짜 미상 그룹을 포함하며, 금액은 할인·환불 등을 조정한 회계 매출이 아닙니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path.name)


EDA 요약 보고서 저장 완료: ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [27]:
# Chapter 06 EDA 함수: completed·미상 그룹 처리 기준을 Notebook에 포함함
"""Chapter 6 EDA 공통 함수 모음.

6장 노트북과 실행 스크립트에서 함께 사용할 탐색적 데이터 분석 함수입니다.
EDA의 핵심 흐름인 질문 정리, 기본 분포 확인, 매출 집계, 결과 저장, 요약 보고서 생성을 함수로 제공합니다.
"""


from pathlib import Path

import pandas as pd


PROCESSED_FILENAMES = {
    "customers": "customers_clean.csv",
    "products": "products_clean.csv",
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
}


def load_processed_sales_data(data_dir: str | Path = "data/processed") -> dict[str, pd.DataFrame]:
    """5장에서 저장한 전처리 데이터 4종을 불러옵니다."""
    base_dir = Path(data_dir)
    data: dict[str, pd.DataFrame] = {}

    for name, filename in PROCESSED_FILENAMES.items():
        path = base_dir / filename
        if not path.exists():
            raise FileNotFoundError(
                f"전처리 파일을 찾을 수 없습니다: {path}\n"
                "먼저 `python scripts/preprocess_data.py`를 실행하세요."
            )
        data[name] = pd.read_csv(path)

    return data


def prepare_eda_data(data: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """EDA 전에 날짜형과 line_total 컬럼을 점검해 보정합니다."""
    prepared = {name: df.copy() for name, df in data.items()}
    prepared["customers"] = prepared["customers"].drop(columns=["name"], errors="ignore")

    customers = prepared["customers"]
    orders = prepared["orders"]
    order_items = prepared["order_items"]

    if "signup_date" in customers.columns:
        customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")

    if "order_date" in orders.columns:
        orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
        if "order_month" not in orders.columns:
            orders["order_month"] = orders["order_date"].dt.to_period("M").astype("string").fillna("날짜 미상")
        if "order_dayofweek" not in orders.columns:
            orders["order_dayofweek"] = orders["order_date"].dt.day_name()

    if "line_total" not in order_items.columns and {"quantity", "unit_price"}.issubset(order_items.columns):
        order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

    prepared["orders"] = orders.loc[orders["order_status"].eq("completed")].copy()
    prepared["order_items"] = order_items.merge(prepared["orders"][["order_id"]], on="order_id", how="inner", validate="many_to_one")
    return prepared


def make_eda_questions() -> pd.DataFrame:
    """6장 기본 EDA 질문과 지표를 표로 반환합니다."""
    return pd.DataFrame(
        {
            "analysis_area": ["고객", "상품", "주문", "매출", "시간", "고객 가치"],
            "question": [
                "고객은 어떤 지역과 연령대에 분포하는가?",
                "어떤 카테고리의 상품이 많은가?",
                "주문 상태와 결제수단 분포는 어떤가?",
                "카테고리별 매출은 어떻게 다른가?",
                "월별 매출과 주문 수는 어떻게 변하는가?",
                "구매 금액이 높은 고객은 누구인가?",
            ],
            "metric": [
                "고객 수, 평균 나이",
                "카테고리별 상품 수",
                "주문 수, 비율",
                "총매출, 매출 비중",
                "월별 매출, 월별 주문 수",
                "고객별 총매출, 주문 횟수",
            ],
            "required_data": [
                "customers",
                "products",
                "orders",
                "orders, order_items, products",
                "orders, order_items",
                "customers, orders, order_items",
            ],
        }
    )


def customer_city_summary(customers: pd.DataFrame) -> pd.DataFrame:
    """도시별 고객 수를 계산합니다."""
    result = customers["city"].value_counts(dropna=False).reset_index()
    result.columns = ["city", "customer_count"]
    result["customer_ratio"] = (result["customer_count"] / result["customer_count"].sum() * 100).round(2)
    return result


def customer_gender_summary(customers: pd.DataFrame) -> pd.DataFrame:
    """성별 고객 수를 계산합니다."""
    result = customers["gender"].value_counts(dropna=False).reset_index()
    result.columns = ["gender", "customer_count"]
    result["customer_ratio"] = (result["customer_count"] / result["customer_count"].sum() * 100).round(2)
    return result


def product_category_summary(products: pd.DataFrame) -> pd.DataFrame:
    """카테고리별 상품 수를 계산합니다."""
    result = products["category"].value_counts(dropna=False).reset_index()
    result.columns = ["category", "product_count"]
    result["product_ratio"] = (result["product_count"] / result["product_count"].sum() * 100).round(2)
    return result


def category_price_summary(products: pd.DataFrame) -> pd.DataFrame:
    """카테고리별 상품 가격 요약 통계를 계산합니다."""
    return (
        products.groupby("category", as_index=False)
        .agg(
            product_count=("product_id", "count"),
            avg_price=("price", "mean"),
            min_price=("price", "min"),
            max_price=("price", "max"),
        )
        .assign(avg_price=lambda df: df["avg_price"].round(0))
        .sort_values("avg_price", ascending=False)
    )


def order_status_summary(orders: pd.DataFrame) -> pd.DataFrame:
    """주문 상태별 주문 수와 비율을 계산합니다."""
    result = orders["order_status"].value_counts(dropna=False).reset_index()
    result.columns = ["order_status", "order_count"]
    result["order_ratio"] = (result["order_count"] / result["order_count"].sum() * 100).round(2)
    return result


def payment_method_summary(orders: pd.DataFrame) -> pd.DataFrame:
    """결제수단별 주문 수와 비율을 계산합니다."""
    result = orders["payment_method"].value_counts(dropna=False).reset_index()
    result.columns = ["payment_method", "order_count"]
    result["order_ratio"] = (result["order_count"] / result["order_count"].sum() * 100).round(2)
    return result


def build_sales_items(products: pd.DataFrame, order_items: pd.DataFrame) -> pd.DataFrame:
    """주문 상세와 상품 데이터를 연결해 카테고리 매출 분석용 데이터를 만듭니다."""
    items = order_items.copy()
    if "line_total" not in items.columns and {"quantity", "unit_price"}.issubset(items.columns):
        items["line_total"] = items["quantity"] * items["unit_price"]
    merged = items.merge(products, on="product_id", how="left", validate="many_to_one")
    merged["category"] = merged["category"].fillna("미연결 상품")
    return merged


def category_sales_summary(products: pd.DataFrame, order_items: pd.DataFrame) -> pd.DataFrame:
    """카테고리별 판매 수량, 매출, 매출 비중을 계산합니다."""
    sales_items = build_sales_items(products, order_items)
    result = (
        sales_items.groupby("category", as_index=False)
        .agg(total_quantity=("quantity", "sum"), total_sales=("line_total", "sum"))
        .sort_values("total_sales", ascending=False)
    )
    result["sales_ratio"] = (result["total_sales"] / result["total_sales"].sum() * 100).round(2)
    return result


def monthly_sales_summary(orders: pd.DataFrame, order_items: pd.DataFrame) -> pd.DataFrame:
    """월별 매출, 주문 수, 평균 주문 금액을 계산합니다."""
    items = order_items.copy()
    if "line_total" not in items.columns and {"quantity", "unit_price"}.issubset(items.columns):
        items["line_total"] = items["quantity"] * items["unit_price"]

    merged = items.merge(orders, on="order_id", how="left", validate="many_to_one")
    merged["order_date"] = pd.to_datetime(merged["order_date"], errors="coerce")
    merged["order_month"] = merged["order_date"].dt.to_period("M").astype("string").fillna("날짜 미상")

    result = (
        merged.groupby("order_month", as_index=False)
        .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
        .sort_values("order_month")
    )
    result["avg_order_value"] = (result["total_sales"] / result["order_count"]).round(0)
    return result


def customer_sales_summary(
    customers: pd.DataFrame,
    orders: pd.DataFrame,
    order_items: pd.DataFrame,
) -> pd.DataFrame:
    """고객별 구매 금액, 주문 횟수, 평균 주문 금액을 계산합니다."""
    items = order_items.copy()
    if "line_total" not in items.columns and {"quantity", "unit_price"}.issubset(items.columns):
        items["line_total"] = items["quantity"] * items["unit_price"]

    order_sales = items.merge(orders, on="order_id", how="left", validate="many_to_one")
    base = order_sales.merge(customers, on="customer_id", how="left", validate="many_to_one")

    base["city"] = base["city"].fillna("미연결 고객")
    group_cols = ["customer_id", "city"]
    if "name" in base.columns:
        group_cols = ["customer_id", "name", "city"]

    result = (
        base.groupby(group_cols, as_index=False)
        .agg(order_count=("order_id", "nunique"), total_sales=("line_total", "sum"))
        .sort_values("total_sales", ascending=False)
    )
    result["avg_order_value"] = (result["total_sales"] / result["order_count"]).round(0)
    return result


def eda_result_summary() -> pd.DataFrame:
    """EDA 결과를 다음 질문으로 연결하는 요약표를 반환합니다."""
    return pd.DataFrame(
        {
            "question": [
                "고객은 어느 도시에 많이 분포하는가?",
                "어떤 카테고리의 상품이 많은가?",
                "카테고리별 매출은 어떻게 다른가?",
                "월별 매출과 주문 수는 어떻게 변하는가?",
                "구매 금액이 높은 고객은 누구인가?",
            ],
            "result_table": [
                "customer_city",
                "product_category",
                "category_sales",
                "monthly_sales",
                "customer_sales",
            ],
            "next_question": [
                "도시별 구매 금액도 차이가 있는가?",
                "상품 수가 많은 카테고리가 매출도 높은가?",
                "매출 차이가 수량 때문인가 단가 때문인가?",
                "특정 월의 매출 변화는 어떤 카테고리 때문인가?",
                "고액 구매 고객은 반복 구매 고객인가?",
            ],
        }
    )


def run_basic_eda(data: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """6장 기본 EDA 결과표를 한 번에 생성합니다."""
    prepared = prepare_eda_data(data)
    customers = prepared["customers"]
    products = prepared["products"]
    orders = prepared["orders"]
    order_items = prepared["order_items"]

    return {
        "questions": make_eda_questions(),
        "customer_city": customer_city_summary(customers),
        "customer_gender": customer_gender_summary(customers),
        "product_category": product_category_summary(products),
        "category_price": category_price_summary(products),
        "order_status": order_status_summary(data["orders"]),
        "payment_method": payment_method_summary(data["orders"]),
        "category_sales": category_sales_summary(products, order_items),
        "monthly_sales": monthly_sales_summary(orders, order_items),
        "customer_sales": customer_sales_summary(customers, orders, order_items),
        "eda_result_summary": eda_result_summary(),
    }


def save_eda_outputs(
    results: dict[str, pd.DataFrame],
    output_dir: str | Path = "reports",
    encoding: str = "utf-8-sig",
) -> list[Path]:
    """EDA 결과표를 CSV 파일로 저장합니다."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    file_map = {
        "category_sales": "ch06_category_sales.csv",
        "monthly_sales": "ch06_monthly_sales.csv",
        "customer_sales": "ch06_customer_sales.csv",
        "eda_result_summary": "ch06_eda_questions.csv",
        "customer_city": "ch06_customer_city.csv",
        "product_category": "ch06_product_category.csv",
    }

    saved_paths: list[Path] = []
    for key, filename in file_map.items():
        path = output_path / filename
        results[key].to_csv(path, index=False, encoding=encoding)
        saved_paths.append(path)

    return saved_paths


def build_eda_report(results: dict[str, pd.DataFrame]) -> str:
    """EDA 결과를 Markdown 보고서 문자열로 생성합니다."""
    questions = results["questions"]
    category_sales = results["category_sales"]
    monthly_sales = results["monthly_sales"]
    customer_sales = results["customer_sales"]
    summary = results["eda_result_summary"]

    return f"""# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전체 고객·상품·주문의 기본 분포와 completed 주문 상세의 금액을 탐색했습니다. 상품·고객 미연결 및 날짜 미상 그룹을 포함하며, 금액은 할인·환불 등을 조정한 회계 매출이 아닙니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
"""



module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)
for key, notebook_table, sort_keys in [
    ('order_status', order_status, ['order_status']),
    ('payment_method', payment_method, ['payment_method']),
    ('category_sales', category_sales, ['category']),
    ('monthly_sales', monthly_sales, ['order_month']),
    ('customer_sales', customer_sales, ['customer_id', 'city'])]:
    left = notebook_table.sort_values(sort_keys).reset_index(drop=True)
    right = module_results[key].sort_values(sort_keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(left, right, check_dtype=False)
    print(key, 'Notebook/함수 일치 PASS')
module_results.keys()


order_status Notebook/함수 일치 PASS
payment_method Notebook/함수 일치 PASS
category_sales Notebook/함수 일치 PASS
monthly_sales Notebook/함수 일치 PASS
customer_sales Notebook/함수 일치 PASS


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [28]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
7,패션,290.0,32940000.0,23.31
5,식품,226.0,25035000.0,17.71
6,전자기기,181.0,21590000.0,15.28
2,뷰티,180.0,17751000.0,12.56
0,도서,174.0,17663000.0,12.50


In [29]:
# Notebook 함수 정의를 별도 프로세스에서 실행해 저장 집계와 비교함
notebook_cells = json.loads((CHAPTER_DIR / 'chapter06.ipynb').read_text(encoding='utf-8'))['cells']
function_source = next(''.join(c['source']) for c in notebook_cells if c['cell_type'] == 'code' and ''.join(c['source']).startswith('# Chapter 06 EDA 함수:')).split('module_data =', 1)[0]
with tempfile.TemporaryDirectory(prefix='ch06-script-') as td:
    script_path = Path(td) / 'eda.py'
    script_path.write_text(function_source + '\nimport sys\nresults = run_basic_eda(load_processed_sales_data(Path(sys.argv[1])))\nsave_eda_outputs(results, Path(sys.argv[2]))\n(Path(sys.argv[2]) / "ch06_eda_summary.md").write_text(build_eda_report(results), encoding="utf-8")\nprint("completed 범위 EDA 스크립트 완료")\n', encoding='utf-8')
    run = subprocess.run([sys.executable, str(script_path), str(PROCESSED_DIR), str(REPORT_DIR / 'script')], capture_output=True, text=True)
if run.returncode: raise RuntimeError(run.stderr)
print(run.stdout)
for key in ['category_sales', 'monthly_sales', 'customer_sales']:
    saved = pd.read_csv(REPORT_DIR / 'script' / f'ch06_{key}.csv')
    pd.testing.assert_frame_equal(saved, module_results[key].reset_index(drop=True), check_dtype=False)
print('Notebook/함수/스크립트 핵심 집계 일치 PASS')


completed 범위 EDA 스크립트 완료

Notebook/함수/스크립트 핵심 집계 일치 PASS


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [30]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액 비교
city_amount = customer_sales_base.groupby('city', as_index=False, dropna=False).agg(
    total_sales=('line_total', 'sum'), buyers=('customer_id', 'nunique'), order_count=('order_id', 'nunique'))
city_comparison = customer_city.merge(city_amount, on='city', how='outer', validate='one_to_one').fillna({'customer_count': 0, 'customer_ratio': 0, 'total_sales': 0, 'buyers': 0, 'order_count': 0})
city_comparison['amount_per_registered_customer'] = city_comparison['total_sales'].div(city_comparison['customer_count'].replace(0, np.nan)).round(0)
city_comparison.sort_values('total_sales', ascending=False)


,city,customer_count,customer_ratio,total_sales,buyers,order_count,amount_per_registered_customer
6,부산,23.0,15.33,24916000.0,16,32,1083304.0
7,서울,18.0,12.00,22211000.0,11,20,1233944.0
2,광주,21.0,14.00,19866000.0,16,28,946000.0
11,인천,16.0,10.67,15723000.0,12,23,982688.0
3,대구,21.0,14.00,14781000.0,12,18,703857.0
9,수원,10.0,6.67,12220000.0,8,13,1222000.0
10,울산,17.0,11.33,11545000.0,9,17,679118.0
8,성남,9.0,6.00,11288000.0,5,13,1254222.0
1,고양,8.0,5.33,7222000.0,6,10,902750.0
0,Unknown,2.0,1.33,820000.0,1,1,410000.0


In [31]:
# 과제 2. 상품 수와 completed 금액을 비교하고 실제 판매 단가를 계산함
category_comparison = product_category.merge(category_sales, on='category', how='outer', validate='one_to_one').fillna({'product_count': 0, 'product_ratio': 0})
category_comparison['weighted_unit_price'] = (category_comparison['total_sales'] / category_comparison['total_quantity']).round(0)
display(category_comparison.sort_values('total_sales', ascending=False))
print('상품 수-금액 Spearman 상관(식별 카테고리만):', category_comparison.loc[category_comparison['product_count'].gt(0), ['product_count','total_sales']].corr(method='spearman').iloc[0,1])


,category,product_count,product_ratio,total_quantity,total_sales,sales_ratio,weighted_unit_price
7,패션,16.0,16.67,290.0,32940000.0,23.31,113586.0
5,식품,16.0,16.67,226.0,25035000.0,17.71,110774.0
6,전자기기,14.0,14.58,181.0,21590000.0,15.28,119282.0
2,뷰티,17.0,17.71,180.0,17751000.0,12.56,98617.0
0,도서,13.0,13.54,174.0,17663000.0,12.50,101511.0
4,스포츠,10.0,10.42,165.0,13237000.0,9.37,80224.0
3,생활용품,10.0,10.42,125.0,9109000.0,6.44,72872.0
1,미연결 상품,0.0,0.00,34.0,4015000.0,2.84,118088.0


상품 수-금액 Spearman 상관(식별 카테고리만): 0.7637626158259733


In [32]:
# 과제 3. 상세 행 평균이 아닌 고유 주문별 금액으로 결제수단별 평균 계산
order_amount = order_sales.groupby(['order_id', 'payment_method'], as_index=False, dropna=False).agg(order_amount=('line_total', 'sum'))
payment_aov = order_amount.groupby('payment_method', as_index=False, dropna=False).agg(
    order_count=('order_id', 'nunique'), total_sales=('order_amount', 'sum'), avg_order_value=('order_amount', 'mean'), median_order_value=('order_amount', 'median'))
payment_aov['avg_order_value'] = payment_aov['avg_order_value'].round(0)
payment_aov.sort_values('avg_order_value', ascending=False)


,payment_method,order_count,total_sales,avg_order_value,median_order_value
1,card,40,37542000.0,938550.0,693000.0
0,bank_transfer,49,39461000.0,805327.0,777000.0
2,kakao_pay,50,36861000.0,737220.0,635000.0
3,naver_pay,40,27476000.0,686900.0,523000.0


In [33]:
# 과제 4. 날짜/상품 미상 그룹도 보존한 월별·카테고리별 금액
monthly_category_base = order_sales.merge(products[['product_id','category']], on='product_id', how='left', validate='many_to_one')
monthly_category_base['category'] = monthly_category_base['category'].fillna('미연결 상품')
monthly_category = monthly_category_base.groupby(['order_month','category'], as_index=False, dropna=False).agg(
    total_sales=('line_total','sum'), order_count=('order_id','nunique'))
monthly_category_pivot = monthly_category.pivot(index='order_month', columns='category', values='total_sales').fillna(0)
monthly_category_pivot


category,도서,미연결 상품,뷰티,생활용품,스포츠,식품,전자기기,패션
order_month,,,,,,,,
2025-07,1040000.0,82000.0,1133000.0,406000.0,295000.0,2123000.0,1319000.0,3163000.0
2025-08,1144000.0,470000.0,1050000.0,983000.0,417000.0,505000.0,2547000.0,2246000.0
2025-09,380000.0,780000.0,125000.0,735000.0,438000.0,528000.0,1157000.0,1156000.0
2025-10,1570000.0,82000.0,661000.0,1334000.0,1426000.0,3088000.0,1781000.0,2687000.0
2025-11,583000.0,420000.0,1581000.0,528000.0,1271000.0,2221000.0,1035000.0,1944000.0
2025-12,34000.0,468000.0,884000.0,489000.0,791000.0,2048000.0,1408000.0,2459000.0
2026-01,74000.0,0.0,376000.0,804000.0,2359000.0,2465000.0,590000.0,1467000.0
2026-02,1827000.0,0.0,275000.0,646000.0,1009000.0,1193000.0,1562000.0,1177000.0
2026-03,1075000.0,140000.0,728000.0,233000.0,364000.0,2792000.0,1378000.0,537000.0


### 월별·카테고리별 결과 해석

- 2026-08은 패션 4,247,000원과 전자기기 3,212,000원이 큰 비중을 차지함
- 카테고리별 고유 주문 수는 여러 카테고리를 산 주문이 중복 포함될 수 있어 합산해 전체 주문 수로 사용하지 않음
- 다음에는 월별 카테고리 금액의 차이가 수량과 단가 중 어디에서 나타나는지 확인할 필요가 있음


In [34]:
# 과제 5. 반복 구매: completed 고유 주문 2회 이상, 고액 단발: 1회·주문 금액 상위 25% 이상(탐색 기준)
high_value_threshold = order_amount['order_amount'].quantile(.75)
customer_segments = customer_sales.copy()
customer_segments['segment'] = np.select([
    customer_segments['order_count'].ge(2),
    customer_segments['order_count'].eq(1) & customer_segments['total_sales'].ge(high_value_threshold)
], ['반복 구매', '고액 단발'], default='기타 단발')
print('고액 기준(주문 금액 75% 분위):', high_value_threshold)
display(customer_segments.groupby('segment', as_index=False).agg(customer_count=('customer_id','nunique'), total_sales=('total_sales','sum')))
print('수량 50 이상 completed 상세:', int(completed_items['quantity'].ge(50).sum()))
print('processed 전체 상품 최고 표시가격:', products['price'].max())


고액 기준(주문 금액 75% 분위): 1098500.0


,segment,customer_count,total_sales
0,고액 단발,8,11554000.0
1,기타 단발,38,21046000.0
2,반복 구매,53,108740000.0


수량 50 이상 completed 상세: 1
processed 전체 상품 최고 표시가격: 9999999.0


In [35]:
totals = pd.Series({'completed 원본': completed_total, '카테고리': category_sales['total_sales'].sum(),
    '월별(날짜 미상 포함)': monthly_sales['total_sales'].sum(), '고객별(미연결 포함)': customer_sales['total_sales'].sum(),
    '도시별': city_comparison['total_sales'].sum(), '결제수단별': payment_aov['total_sales'].sum(),
    '월별·카테고리별': monthly_category['total_sales'].sum()}, name='amount')
assert totals.eq(completed_total).all()
display(totals.to_frame())
print('총합 검증 PASS')
print('카테고리 미연결 제외 시 차이:', completed_total - category_sales.loc[category_sales['category'].ne('미연결 상품'),'total_sales'].sum())
print('날짜 미상 제외 시 차이:', completed_total - monthly_sales.loc[monthly_sales['order_month'].ne('날짜 미상'),'total_sales'].sum())


,amount
completed 원본,141340000.0
카테고리,141340000.0
월별(날짜 미상 포함),141340000.0
고객별(미연결 포함),141340000.0
도시별,141340000.0
결제수단별,141340000.0
월별·카테고리별,141340000.0


총합 검증 PASS
카테고리 미연결 제외 시 차이: 4015000.0
날짜 미상 제외 시 차이: 1903000.0


## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


### 과제 6. 답할 수 없는 질문을 검증 가능한 질문으로 바꾸기

- 광고 효과와 상품 선호 이유를 현재 데이터로 계산 가능한 질문으로 바꾸도록 ChatGPT 새 대화에서 실제 요청함

```text
온라인 쇼핑몰 데이터로 EDA를 진행 중입니다.

사용 가능한 컬럼:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

completed 주문 상세 금액은 141,340,000원입니다.
상품 미연결 금액 4,015,000원과 날짜 미상 금액 1,903,000원을 포함합니다.

현재 데이터로는 답하기 어려운 질문:
1. 광고가 매출을 높였는가?
2. 고객은 왜 특정 상품을 선호하는가?

이 질문들을 현재 컬럼으로 검증 가능한 EDA 질문 3개로 바꿔주세요.
각 질문의 필요한 데이터, 지표, 계산·검증 방법을 간단히 표로 정리해주세요.
관찰과 원인 가설을 구분하고, 없는 정보를 추측하지 마세요.
```

- 실제 응답은 기간별 금액·주문 건수·객단가, 카테고리별 금액·수량·비중, 연령대·성별 구매 금액·주문 빈도 세 가지를 제안함
- 앞의 두 제안은 completed·미상 그룹·고유 주문 수 기준을 명시해 수정 후 사용함
- 연령대·성별 제안은 나이 대체값과 이상값, 연령 구간 정의를 먼저 확인해야 하므로 후속 분석으로 보류함
- 응답의 관찰 예시는 실제 계산 결과로 취급하지 않고 Notebook 출력으로 확인한 사실과 구분함


In [36]:
# 실제 LLM 응답의 제안과 현재 계산 결과를 대조함
proposal_checks = pd.DataFrame([
    {'question': '기간별 금액·주문 건수·객단가', 'columns_exist': {'order_date','order_id','order_status'}.issubset(orders.columns) and {'line_total'}.issubset(completed_items.columns), 'computed_rows': len(monthly_sales), 'decision': '수정 후 사용'},
    {'question': '카테고리별 금액·수량·비중', 'columns_exist': {'product_id','category'}.issubset(products.columns) and {'product_id','quantity','line_total'}.issubset(completed_items.columns), 'computed_rows': len(category_sales), 'decision': '수정 후 사용'},
    {'question': '연령대·성별 구매 금액·주문 빈도', 'columns_exist': {'customer_id','age','gender'}.issubset(customers.columns) and {'customer_id','order_id'}.issubset(orders.columns), 'computed_rows': 0, 'decision': '보류: 연령 기준·대체값·이상값 확인 필요'}])
assert proposal_checks['columns_exist'].all()
display(proposal_checks)
# 응답의 중복 집계 주의를 실제 상세 행 기준으로 확인함
product_missing = monthly_category_base['category'].eq('미연결 상품')
date_missing = monthly_category_base['order_date'].isna()
print('상품 미연결·날짜 미상 중복 상세:', int((product_missing & date_missing).sum()))
print('중복 금액:', monthly_category_base.loc[product_missing & date_missing,'line_total'].sum())
print('두 조건 중 하나 이상 해당하는 고유 상세 금액:', monthly_category_base.loc[product_missing | date_missing,'line_total'].sum())
# 미래일은 자동 삭제하지 않고 날짜 확인이 필요한 값으로 따로 집계함
future_mask = order_sales['order_date'].gt(pd.Timestamp('2026-10-19'))
print('작성일 이후 주문:', order_sales.loc[future_mask,'order_id'].nunique(), '상세:', int(future_mask.sum()), '금액:', order_sales.loc[future_mask,'line_total'].sum())
print('관측된 completed 날짜:', order_sales['order_date'].min(), '~', order_sales['order_date'].max())
print('날짜 미상 상세의 고유 주문:', order_sales.loc[order_sales['order_date'].isna(),'order_id'].nunique())
print('상세 없는 completed 주문:', int((~completed_orders['order_id'].isin(completed_items['order_id'])).sum()))


,question,columns_exist,computed_rows,decision
0,기간별 금액·주문 건수·객단가,True,16,수정 후 사용
1,카테고리별 금액·수량·비중,True,8,수정 후 사용
2,연령대·성별 구매 금액·주문 빈도,True,0,보류: 연령 기준·대체값·이상값 확인 필요


상품 미연결·날짜 미상 중복 상세: 0
중복 금액: 0.0
두 조건 중 하나 이상 해당하는 고유 상세 금액: 5918000.0
작성일 이후 주문: 1 상세: 3 금액: 909000.0
관측된 completed 날짜: 2025-07-02 00:00:00 ~ 2027-01-15 00:00:00
날짜 미상 상세의 고유 주문: 2
상세 없는 completed 주문: 1


## 나의 추가 EDA

### 선택한 질문

- 수량 50 이상인 상세를 제외해도 카테고리별 금액 순위가 유지되는가?

### 선택 이유

- Chapter 05에서 큰 수량을 확인했고 패션 금액이 가장 높아 대량 구매 상세가 순위에 영향을 주는지 확인하고자 함

### 분석 범위

- completed 주문에 연결된 상세를 대상으로 수량 50 이상인 행을 포함한 경우와 제외한 경우를 비교함
- 주문 전체가 아닌 해당 상세 행만 제외하며 원본과 processed 파일은 바꾸지 않음
- 날짜 미상과 미연결 상품 금액은 합계에 포함하고 미연결 상품은 실제 카테고리 순위에서 제외함

### 필요한 데이터와 컬럼

- orders: order_id, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price, line_total
- products: product_id, category

### 사용할 지표

- 카테고리별 상세 금액 합계, 전체 금액 대비 비중, 금액 순위
- 제외한 상세 수와 금액, 제외 전후 총액 차이

### 검증 방법

- 앞에서 행 수와 연결 관계를 확인한 sales_items를 사용함
- 제외 전 총액에서 제외 금액을 빼면 제외 후 총액과 같은지 확인함
- 카테고리별 금액 합계와 상세 원본 합계를 비교하고 실제 카테고리 7개의 순위 변화를 확인함


In [37]:
# 수량 50 이상인 상세의 포함 여부에 따른 카테고리 금액 비교
large_quantity = sales_items['quantity'].ge(50)
excluded_items = sales_items.loc[large_quantity].copy()
remaining_items = sales_items.loc[~large_quantity].copy()
display(excluded_items[['order_item_id', 'order_id', 'category', 'quantity', 'unit_price', 'line_total']])

before_amount = sales_items.groupby('category')['line_total'].sum()
after_amount = remaining_items.groupby('category')['line_total'].sum().reindex(before_amount.index, fill_value=0)
quantity_comparison = pd.DataFrame({'포함 금액': before_amount, '제외 후 금액': after_amount})
quantity_comparison['금액 차이'] = before_amount - after_amount
quantity_comparison['포함 비중(%)'] = (before_amount / before_amount.sum() * 100).round(2)
quantity_comparison['제외 후 비중(%)'] = (after_amount / after_amount.sum() * 100).round(2)
known_categories = quantity_comparison.index != '미연결 상품'
quantity_comparison['포함 순위'] = before_amount.where(known_categories).rank(ascending=False, method='min').astype('Int64')
quantity_comparison['제외 후 순위'] = after_amount.where(known_categories).rank(ascending=False, method='min').astype('Int64')
display(quantity_comparison.sort_values('포함 금액', ascending=False))

before_total = sales_items['line_total'].sum()
after_total = remaining_items['line_total'].sum()
excluded_total = excluded_items['line_total'].sum()
assert len(sales_items) == len(remaining_items) + len(excluded_items)
assert before_total == completed_total == before_amount.sum()
assert after_total == after_amount.sum()
assert before_total - excluded_total == after_total
print(f'상세 행 수: {len(sales_items)} → {len(remaining_items)}, 제외 {len(excluded_items)}행')
print(f'총액: {before_total:,.0f}원 → {after_total:,.0f}원, 제외 {excluded_total:,.0f}원')
print('총액과 행 수 검증 PASS')
print('실제 카테고리 순위 유지:', quantity_comparison.loc[known_categories, '포함 순위'].equals(quantity_comparison.loc[known_categories, '제외 후 순위']))


,order_item_id,order_id,category,quantity,unit_price,line_total
432,738,290,패션,50.0,96000.0,4800000.0


,포함 금액,제외 후 금액,금액 차이,포함 비중(%),제외 후 비중(%),포함 순위,제외 후 순위
category,,,,,,,
패션,32940000.0,28140000.0,4800000.0,23.31,20.61,1,1
식품,25035000.0,25035000.0,0.0,17.71,18.34,2,2
전자기기,21590000.0,21590000.0,0.0,15.28,15.81,3,3
뷰티,17751000.0,17751000.0,0.0,12.56,13.00,4,4
도서,17663000.0,17663000.0,0.0,12.50,12.94,5,5
스포츠,13237000.0,13237000.0,0.0,9.37,9.69,6,6
생활용품,9109000.0,9109000.0,0.0,6.44,6.67,7,7
미연결 상품,4015000.0,4015000.0,0.0,2.84,2.94,<NA>,<NA>


상세 행 수: 446 → 445, 제외 1행
총액: 141,340,000원 → 136,540,000원, 제외 4,800,000원
총액과 행 수 검증 PASS
실제 카테고리 순위 유지: True


### 결과 관찰

- completed 상세 446행 중 수량 50 이상은 패션 상세 1행이며 수량 50개, 금액 4,800,000원임
- 해당 행을 제외하면 총액은 141,340,000원에서 136,540,000원으로 줄고 제외 금액과 총액 차이가 일치함
- 패션 금액은 32,940,000원에서 28,140,000원으로 줄지만 1위를 유지하며 실제 카테고리 7개의 순위도 모두 같음
- 패션 비중은 23.31%에서 20.61%로 낮아지고 2위 식품과의 차이는 7,905,000원에서 3,105,000원으로 줄어듦

### 나의 해석과 판단

- 수량 50인 상세 한 행 때문에 패션이 1위가 된 것은 아님을 확인함
- 순위는 같아도 금액 비중과 2위와의 차이는 줄어들어 순위만 비교하면 변화가 잘 보이지 않음

### 현재 분석의 한계

- 수량 50은 앞에서 확인한 큰 값을 비교하기 위한 기준이며 오류를 판정하는 기준은 아님
- 실제 대량 구매일 수 있어 이 결과만으로 해당 행을 삭제할 근거는 없음
- 다른 수량 기준이나 고액 상세까지 제외한 경우는 확인하지 않았으며 상품 미연결 문제도 남아 있음

### 다음 질문

- 수량 대신 상세 금액이 큰 행을 기준으로 비교해도 패션의 순위와 비중이 비슷하게 나타나는가?


# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

> 이 내용을 `chapter06.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
이름: 장태원  
GitHub ID: pmsky1  
작성일: 2026-10-19  
최종 Notebook URL: https://github.com/pmsky1/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb  

## 1. 내가 정한 EDA 질문
| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 상품 수가 많은 카테고리가 completed 금액도 높은가? | 전체 상품 수와 completed 상세, 상품 미연결 포함 | 상품 수·판매 수량·line_total 합계·실제 판매 단가 | 상품 키 유일성·병합 행 수·원본 총합 확인 |
| 월별 completed 금액 차이는 주문 수와 평균 주문 금액에서 어떻게 나타나는가? | completed 상세, 날짜 미상 별도, 추가 기간 필터 없음 | 월별 line_total·고유 주문 수·평균 주문 금액 | 날짜 재변환·미상 금액·월별 총합 확인 |
| 결제수단별 평균 주문 금액은 어떻게 다른가? | completed 고유 주문별 상세 합산 | 결제수단별 총금액/고유 주문 수·중앙값 | 주문별 금액을 먼저 계산하고 총합·고유 주문 수 확인 |
| 도시별 고객 수와 completed 금액은 비슷한가? | 전체 등록 고객과 completed 구매 고객·미연결 고객 별도 | 등록 고객 수·구매 고객 수·금액·등록 고객당 금액 | 고객 키·미연결 고객 금액·도시별 총합 확인 |

- 질문 1: products에서 상품 수를 세고 orders·order_items에서 completed 수량과 금액을 합산함, 판매 단가는 금액 합계/수량 합계로 계산함
- 질문 2: orders·order_items를 연결해 월별 금액과 고유 주문 수를 구함, 평균 주문 금액은 월 금액/고유 주문 수임
- 질문 3: orders·order_items에서 주문별 금액을 먼저 합한 뒤 결제수단별 평균과 중앙값을 구함
- 질문 4: customers의 도시별 고객 수와 orders·order_items의 completed 금액을 비교함, 고객당 금액은 도시 금액/등록 고객 수임

### 왜 이 질문을 선택했는가?

- 상품 수가 많은 카테고리에서 구매도 많이 일어나는지 확인하고 싶어 선택함
- 주문일·결제수단·상태와 상세 금액이 있어 월별·결제수단별 차이를 계산할 수 있음
- 주문 수는 상세 행 수가 아닌 고유 order_id 수로 세고, 상품 표시가격과 실제 판매 단가를 구분함

## 2. 핵심 집계 결과
- completed 원본 합계: 141,340,000원, 상세 446행·상세가 있는 고유 주문 179건
- completed 주문 테이블은 180건이며 상세가 없는 1건은 금액 합계와 평균 주문 금액 계산 대상에서 제외함, 평균의 분모는 상세가 있는 179건임
- 카테고리별 합계: 141,340,000원, 미연결 상품 4,015,000원 포함
- 월별 합계: 141,340,000원, 날짜 미상 1,903,000원 포함
- 총합 일치 여부: 원본·카테고리·월·고객·도시·결제수단·월별 카테고리 금액 모두 일치함

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰

- 상품 수는 뷰티가 17개로 가장 많고 completed 금액은 패션이 32,940,000원으로 가장 큼
- 패션 판매 수량은 290개, 실제 판매 단가는 금액/수량 기준 약 113,586원이며 전자기기 약 119,282원보다 낮음
- 결제수단별 평균 주문 금액은 card 938,550원, naver_pay 686,900원이며 card 중앙값은 693,000원임
- 부산은 등록 고객 23명·금액 24,916,000원으로 가장 크지만 등록 고객당 금액은 성남 약 1,254,222원이 부산 약 1,083,304원보다 높음

### 나의 해석과 판단

- 상품 수 1위와 금액 1위가 달라 상품 수만으로 금액 차이를 설명하기 어려움
- 카드 평균과 중앙값의 차이가 있어 고액 주문 영향을 추가로 확인할 필요가 있음
- 미연결 상품·날짜 미상을 제외하면 합계가 달라지므로 총금액에는 포함하고 해당 그룹을 별도로 표시함

### 업무·분석적 의미

- 패션 금액이 높은 이유를 보려면 상품별 판매 수량과 단가를 더 확인할 필요가 있음
- 도시나 결제수단별 금액 차이를 판단하기 전에 고객 수와 주문 구성이 다른지 확인할 필요가 있음

### 한계와 추가 확인 사항

- 금액은 completed 상세의 quantity×unit_price이며 환불·할인·세금·원가를 반영한 확정 회계 매출이나 이익이 아님
- 관측된 completed 주문일은 2025-07-02~2027-01-15이고 날짜 미상 주문도 있어 단순 월별 증감으로 추세·계절성을 단정할 수 없음
- 2027년 1월은 작성일 이후 값이므로 날짜 오류나 샘플 데이터 설정 여부를 확인해야 하며 임의 삭제하지 않음
- 미연결 상품·고객이 남아 있음, 수량 50 이상 상세를 제외해도 순위는 유지됐지만 다른 고액 상세와 표시가격 9,999,999원의 영향은 추가 확인이 필요함
- 상품 표시가격 평균과 실제 주문 단가는 다르며 등록 고객당 금액은 구매 고객당 평균과 다른 지표임

## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: 뷰티 상품 수 17개가 가장 많지만 금액은 패션 32,940,000원이 가장 높음
- 가설: 패션의 판매 수량이나 실제 판매 단가 구성이 금액 차이에 영향을 줄 수 있음
- 추가 검증: 패션은 전자기기보다 판매 단가가 낮고 수량이 많은 것을 확인함, 수량 50 이상인 상세 1행을 제외해도 패션 1위는 유지됐으며 다른 고액 상세의 영향은 추가 확인이 필요함

### 사례 2
- 관찰: 날짜가 확인된 월 중 2026-08 금액 16,980,000원이 가장 높고 고유 주문은 15건임
- 가설: 월별 금액 차이는 주문 수와 주문당 금액 중 하나 또는 둘 모두의 변화와 관련될 수 있음
- 추가 검증: 7월은 8건·평균 1,120,750원, 8월은 15건·평균 1,132,000원으로 평균보다 주문 수 차이가 컸음, 다음에는 고액 주문과 월별 수집 기간의 영향을 확인할 필요가 있음

### 사례 3
- 관찰: completed 주문 2회 이상인 고객 ID는 53개, 주문 1회·고액 기준 이상은 8개임
- 가설: 높은 총구매 금액은 반복 구매 누적과 단발 고액 주문에서 서로 다르게 나타날 수 있음
- 추가 검증: 고액 기준은 주문 금액 상위 25%인 1,098,500원으로 정함, 기준을 바꿨을 때 결과가 달라지는지와 미연결 고객 ID를 확인할 필요가 있음

## 4. LLM 질문 확장
- Safe Context: 이름이 없는 네 테이블의 컬럼, completed 총액 141,340,000원, 상품 미연결 4,015,000원과 날짜 미상 1,903,000원만 제공함
- Prompt: 위 과제 6의 실제 요청문으로 광고 효과·선호 이유를 검증 가능한 EDA 질문 3개로 바꾸도록 요청함
- LLM 제안 요약: 기간별 금액·주문 건수·객단가, 카테고리별 금액·수량·비중, 연령대·성별 구매 금액·주문 빈도를 제안함
- 실제 사용 여부: 기간별·카테고리별 제안은 수정 후 사용, 연령대·성별 제안은 보류
- 판단 이유: 기간별·카테고리별 지표는 실제 계산 결과가 있으며 연령 분석은 나이 대체값·이상값과 구간 기준을 추가 확인해야 함
- 사람이 수정한 내용: 기간은 월별로 정하고 completed 범위, 날짜 미상·미연결 그룹 보존, 고유 주문 수 분모를 명시함
- 응답의 관찰 문장은 설명용 예시이며 실제 결과로 인용하지 않음
- 상품 미연결과 날짜 미상의 중복 상세는 0행·0원으로 확인했고 두 조건 중 하나 이상 해당하는 금액은 5,918,000원임, 전체 금액에서 실제로 누락된 금액을 뜻하지 않음

![LLM EDA 질문 확장](images/step04_llm.png)

![LLM 관찰·가설 구분과 집계 주의사항](images/step04_llm_review.png)

## 5. 다음 분석 우선순위
1. 날짜 미상·2027년 주문일과 미연결 상품·고객의 원본 확인
2. 수량 기준 비교에 이어 상세 금액이 큰 행의 포함 여부에 따른 결과 비교
3. 월별 카테고리·고유 주문 수·평균 주문 금액 및 반복 구매 분포 추가 비교

### 가장 먼저 확인하고 싶은 이유

- 어느 월·상품·고객의 금액인지 불명확한 값이 있어 원본 날짜와 ID부터 확인할 필요가 있음

### 현재 데이터만으로 단정할 수 없는 것

- 광고·프로모션 효과, 상품 선호 이유, 고객 만족·이탈, 재고·원가·수익성은 필요한 정보가 없어 단정할 수 없음
- 총합은 맞지만 날짜와 ID 문제는 남아 있음, 현재 차이만으로 원인이나 장기 추세를 단정할 수 없음

### 이번 장에서 EDA에 대해 새롭게 이해한 점

- 총합이 맞는지 확인하는 것과 큰 값이 결과에 미치는 영향을 확인하는 것은 별개임을 확인함
- 수량 50인 상세를 제외해도 순위는 같았지만 비중은 달라져 질문에 맞게 여러 지표를 함께 봐야 함
- 관찰한 차이를 설명할 수 있는 원인과 데이터로 직접 확인한 결과를 구분할 필요가 있음

## 최종 체크
- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [ ] 최종 Notebook URL을 제출합니다.
